# DistilBERT Fine-Tuning on Combined Data
This notebook fine-tunes DistilBERT on `Backend/app/db/Combined Data.csv` for mental health classification.

## Install and Import Libraries
Install required packages and import PyTorch, transformers, datasets, pandas, and evaluation utilities.

In [8]:
!python3 -m pip install -q 'transformers[torch]' datasets accelerate evaluate torch pandas scikit-learn

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import evaluate

print("Libraries loaded successfully.")

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


## Load combined data.csv
Read the combined CSV file into a pandas DataFrame and inspect columns, class distribution, and sample rows.

In [2]:
path = '../app/db/Combined Data.csv'
df = pd.read_csv(path)
print(f"Dataset shape: {df.shape}")
print(df.columns.tolist())
print(df["status"].value_counts())
print(df.head(5))

Dataset shape: (53043, 3)
['Unnamed: 0', 'statement', 'status']
status
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64
   Unnamed: 0                                          statement   status
0           0                                         oh my gosh  Anxiety
1           1  trouble sleeping, confused mind, restless hear...  Anxiety
2           2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3           3  I've shifted my focus to something else but I'...  Anxiety
4           4  I'm restless and restless, it's been a month n...  Anxiety


## Preprocess Text and Labels
Clean and prepare text data, map labels to integers, and split into train/validation sets.

In [3]:
# Drop rows with missing statement values
df = df.dropna(subset=["statement", "status"]).reset_index(drop=True)

# Label mapping
label_names = sorted(df["status"].unique())
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df["status"].map(label2id)
print("Label mapping:", label2id)

# Train / validation split
train_df, valid_df = train_test_split(df[["statement", "label"]], test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {valid_df.shape}")

Label mapping: {'Anxiety': 0, 'Bipolar': 1, 'Depression': 2, 'Normal': 3, 'Personality disorder': 4, 'Stress': 5, 'Suicidal': 6}
Train shape: (42144, 2)
Validation shape: (10537, 2)


## Create Hugging Face Dataset
Convert the pandas DataFrame to a Hugging Face Dataset and apply tokenization to the text fields.

In [4]:
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True))
})

model_name = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

max_length = 128

def tokenize_fn(example):
    return tokenizer(example["statement"], padding="max_length", truncation=True, max_length=max_length)

encoded_datasets = raw_datasets.map(tokenize_fn, batched=True)
encoded_datasets = encoded_datasets.remove_columns(["statement"])
encoded_datasets.set_format(type="torch")

print(encoded_datasets)

Map: 100%|██████████| 10537/10537 [00:00<00:00, 14805.79 examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 42144
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 10537
    })
})


## Load DistilBERT Tokenizer and Model
Load the DistilBERT tokenizer and sequence classification model from transformers, set num_labels.

In [5]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)
print("Model loaded with", len(label_names), "labels.")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded with 7 labels.


## Set Up Training Arguments
Configure Trainer arguments like batch size, learning rate, epochs, evaluation strategy, and output directory.

In [6]:
output_dir = "../fine_tuned_distilbert"
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)
print(training_args)

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,

## Fine-Tune DistilBERT
Initialize Trainer with model, dataset, tokenizer, and metrics, then run fine-tuning.

In [7]:
metric_accuracy = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")
metric_precision = evaluate.load("precision")
metric_recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = metric_accuracy.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")
    precision = metric_precision.compute(predictions=predictions, references=labels, average="weighted")
    recall = metric_recall.compute(predictions=predictions, references=labels, average="weighted")
    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"],
        "precision": precision["precision"],
        "recall": recall["recall"],
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/var/folders/y9/qcfn2ghd0898g39ll1kk2bnw0000gn/T/ipykernel_25569/4220363078.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/homebrew/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.494100,0.484128,0.804878,0.800764,0.806382,0.804878
2,0.399000,0.442369,0.822530,0.821648,0.822722,0.822530
3,0.287600,0.461075,0.826042,0.826440,0.827326,0.826042


/opt/homebrew/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/homebrew/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=7902, training_loss=0.4353897796646307, metrics={'train_runtime': 6216.439, 'train_samples_per_second': 20.338, 'train_steps_per_second': 1.271, 'total_flos': 4187402885357568.0, 'train_loss': 0.4353897796646307, 'epoch': 3.0})

## Evaluate and Save Model
Evaluate the trained model on validation data, print metrics, and save the final model artifacts.

In [8]:
eval_results = trainer.evaluate()
print("Evaluation results:")
print(eval_results)



/opt/homebrew/lib/python3.11/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluation results:
{'eval_loss': 0.4610751271247864, 'eval_accuracy': 0.8260415678086742, 'eval_f1': 0.8264401876617913, 'eval_precision': 0.8273261849612333, 'eval_recall': 0.8260415678086742, 'eval_runtime': 83.538, 'eval_samples_per_second': 126.134, 'eval_steps_per_second': 3.95, 'epoch': 3.0}


In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")